# LSEG DATA AUDIT

**Project setup and file inventory**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Define project directories
PROJECT_DIR = Path(r"C:\UTAR - Master\Master Project\Project\Main Project")

RAW_DIR = PROJECT_DIR / "data" / "raw_lseg"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
RESULTS_DIR = PROJECT_DIR / "results"
REFERENCE_DIR = PROJECT_DIR / "references"
PAPER_DIR = PROJECT_DIR / "conference_paper"

# Create missing directories
for folder in [
    PROCESSED_DIR,
    NOTEBOOK_DIR,
    RESULTS_DIR,
    REFERENCE_DIR,
    PAPER_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# Verify raw data directory
assert RAW_DIR.exists(), f"Directory not found: {RAW_DIR}"

# Identify Excel and CSV files
SUPPORTED_EXTENSIONS = {".xlsx", ".xls", ".csv"}

raw_files = sorted([
    file for file in RAW_DIR.iterdir()
    if file.is_file()
    and file.suffix.lower() in SUPPORTED_EXTENSIONS
    and not file.name.startswith("~$")
])

# Create file inventory
inventory = pd.DataFrame([
    {
        "Filename": file.name,
        "Extension": file.suffix.lower(),
        "Size_KB": round(file.stat().st_size / 1024, 2)
    }
    for file in raw_files
])

print("=== LSEG DATA AUDIT ===")
print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Files detected:", len(raw_files))

display(inventory)

# Validate expected number of files
assert len(raw_files) == 13, (
    f"Expected 13 files, found {len(raw_files)}"
)

# Save inventory
inventory.to_csv(
    RESULTS_DIR / "lseg_raw_file_inventory.csv",
    index=False
)

print("\nAll 13 LSEG files detected successfully!")

=== LSEG DATA AUDIT ===
Project directory: C:\UTAR - Master\Master Project\Main Project
Raw data directory: C:\UTAR - Master\Master Project\Main Project\data\raw_lseg
Files detected: 13


,Filename,Extension,Size_KB
0,Affin Bank_Price History_20261102_1512.xlsx,.xlsx,249.98
1,Alliance Bank_Price History_20261102_1511.xlsx,.xlsx,268.90
2,Ambank_Price History_20261102_1510.xlsx,.xlsx,240.78
3,BIMB_Price History_20261102_1513.xlsx,.xlsx,237.64
4,CIMB_Price History_20261102_1505.xlsx,.xlsx,270.21
5,DBS_Price History_20261102_1516.xlsx,.xlsx,331.12
6,Hong Leong_Price History_20261102_1508.xlsx,.xlsx,244.69
7,Maybank_Price History_20261102_1459.xlsx,.xlsx,248.04
8,MBSB_Price History_20261102_1514.xlsx,.xlsx,228.80
9,OCBC_Price History_20261102_1518.xlsx,.xlsx,290.87



All 13 LSEG files detected successfully!


**Locate historical price tables in LSEG exports**

In [8]:
from openpyxl import load_workbook
import pandas as pd

inspection_results = []

for file in raw_files:
    workbook = load_workbook(
        file,
        read_only=True,
        data_only=True
    )

    for sheet in workbook.worksheets:
        header_row = None
        header_values = None

        for row_number, row in enumerate(
            sheet.iter_rows(values_only=True),
            start=1
        ):
            first_value = row[0]

            if (
                isinstance(first_value, str)
                and "Exchange Date" in first_value
            ):
                header_row = row_number
                header_values = row
                break

        inspection_results.append({
            "Filename": file.name,
            "Worksheet": sheet.title,
            "Total_Rows": sheet.max_row,
            "Header_Row": header_row,
            "Headers": (
                " | ".join(
                    str(value)
                    for value in header_values
                    if value is not None
                )
                if header_values is not None
                else None
            )
        })

    workbook.close()

inspection_df = pd.DataFrame(inspection_results)

print("=== LSEG HISTORICAL TABLE INSPECTION ===")
display(inspection_df)

# Verify historical headers were found
assert inspection_df["Header_Row"].notna().all(), (
    "Historical table header missing in one or more worksheets"
)

inspection_df.to_csv(
    RESULTS_DIR / "lseg_historical_table_inspection.csv",
    index=False
)

print("\nHistorical table headers located successfully.")

=== LSEG HISTORICAL TABLE INSPECTION ===


,Filename,Worksheet,Total_Rows,Header_Row,Headers
0,Affin Bank_Price History_20261102_1512.xlsx,Sheet 1,2924,28,Exchange Date | Close | Net | %Chg | Open | Lo...
1,Alliance Bank_Price History_20261102_1511.xlsx,Sheet 1,2928,32,Exchange Date | Close | Net | %Chg | Open | Lo...
2,Ambank_Price History_20261102_1510.xlsx,Sheet 1,2929,33,Exchange Date | Close | Net | %Chg | Open | Lo...
3,BIMB_Price History_20261102_1513.xlsx,Sheet 1,2929,33,Exchange Date | Close | Net | %Chg | Open | Lo...
4,CIMB_Price History_20261102_1505.xlsx,Sheet 1,2926,30,Exchange Date | Close | Net | %Chg | Open | Lo...
5,DBS_Price History_20261102_1516.xlsx,Sheet 1,2998,30,Exchange Date | Close | Net | %Chg | Open | Lo...
6,Hong Leong_Price History_20261102_1508.xlsx,Sheet 1,2927,31,Exchange Date | Close | Net | %Chg | Open | Lo...
7,Maybank_Price History_20261102_1459.xlsx,Sheet 1,2925,29,Exchange Date | Close | Net | %Chg | Open | Lo...
8,MBSB_Price History_20261102_1514.xlsx,Sheet 1,2928,32,Exchange Date | Close | Net | %Chg | Open | Lo...
9,OCBC_Price History_20261102_1518.xlsx,Sheet 1,2997,29,Exchange Date | Close | Net | %Chg | Open | Lo...



Historical table headers located successfully.


**Extract and validate LSEG historical OHLCV data**

In [10]:
from openpyxl import load_workbook
from datetime import datetime, date
import pandas as pd
import numpy as np
import re

# Define stock identifiers
STOCK_INFO = {
    "Affin Bank": ("5185.KL", "Malaysia"),
    "Alliance Bank": ("2488.KL", "Malaysia"),
    "Ambank": ("1015.KL", "Malaysia"),
    "BIMB": ("5258.KL", "Malaysia"),
    "CIMB": ("1023.KL", "Malaysia"),
    "DBS": ("D05.SI", "Singapore"),
    "Hong Leong": ("5819.KL", "Malaysia"),
    "Maybank": ("1155.KL", "Malaysia"),
    "MBSB": ("1171.KL", "Malaysia"),
    "OCBC": ("O39.SI", "Singapore"),
    "Public Bank": ("1295.KL", "Malaysia"),
    "RHB": ("1066.KL", "Malaysia"),
    "UOB": ("U11.SI", "Singapore")
}

REQUIRED_COLUMNS = [
    "Exchange Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Volume"
]

def parse_lseg_date(value):
    if isinstance(value, (datetime, date, pd.Timestamp)):
        return pd.Timestamp(value).normalize()

    if isinstance(value, str):
        return pd.to_datetime(
            value.strip(),
            format="%d-%b-%Y",
            errors="coerce"
        )

    return pd.NaT


def parse_lseg_number(value):
    if value is None:
        return np.nan

    try:
        return float(str(value).replace(",", "").strip())
    except (ValueError, TypeError):
        return np.nan


all_stocks = []
audit_records = []

# Extract historical observations from each workbook
for file in raw_files:

    stock_name = file.name.split("_Price History_")[0]
    ticker, country = STOCK_INFO[stock_name]

    workbook = load_workbook(
        file,
        read_only=True,
        data_only=True
    )

    sheet = workbook.worksheets[0]

    header_row = None
    header_map = {}
    lseg_ric = None

    # Identify the LSEG instrument code and historical header
    for row_number, row in enumerate(
        sheet.iter_rows(values_only=True),
        start=1
    ):
        first_value = row[0]

        if isinstance(first_value, str):
            text = first_value.strip()

            if (
                lseg_ric is None
                and re.fullmatch(
                    r"[A-Za-z0-9._-]+\.(KL|SI)",
                    text
                )
            ):
                lseg_ric = text

            if "Exchange Date" in text:
                header_row = row_number

                header_map = {
                    str(value).strip(): index
                    for index, value in enumerate(row)
                    if value is not None
                }
                break

    assert header_row is not None, (
        f"Historical header missing: {file.name}"
    )

    assert all(
        column in header_map
        for column in REQUIRED_COLUMNS
    ), f"Required columns missing: {file.name}"

    records = []

    # Read valid daily records below the historical header
    for row in sheet.iter_rows(
        min_row=header_row + 1,
        values_only=True
    ):

        raw_date = row[header_map["Exchange Date"]]
        parsed_date = parse_lseg_date(raw_date)

        if pd.isna(parsed_date):
            continue

        record = {
            "Date": parsed_date,
            "Open": parse_lseg_number(
                row[header_map["Open"]]
            ),
            "High": parse_lseg_number(
                row[header_map["High"]]
            ),
            "Low": parse_lseg_number(
                row[header_map["Low"]]
            ),
            "Close": parse_lseg_number(
                row[header_map["Close"]]
            ),
            "Volume": parse_lseg_number(
                row[header_map["Volume"]]
            ),
            "Ticker": ticker,
            "Name": stock_name,
            "Country": country,
            "LSEG_RIC": lseg_ric
        }

        records.append(record)

    workbook.close()

    stock_df = pd.DataFrame(records)

    assert not stock_df.empty, (
        f"No historical observations extracted: {file.name}"
    )

    stock_df = (
        stock_df
        .sort_values("Date")
        .reset_index(drop=True)
    )

    price_cols = ["Open", "High", "Low", "Close"]

    missing_ohlcv = stock_df[
        price_cols + ["Volume"]
    ].isna().any(axis=1)

    invalid_prices = (
        (stock_df[price_cols] <= 0)
        .any(axis=1)
    )

    invalid_volume = stock_df["Volume"] < 0

    invalid_high_low = (
        stock_df["High"] < stock_df["Low"]
    )

    invalid_ohlc = (
        (stock_df["High"] < stock_df["Open"])
        | (stock_df["High"] < stock_df["Close"])
        | (stock_df["Low"] > stock_df["Open"])
        | (stock_df["Low"] > stock_df["Close"])
    )

    duplicate_dates = stock_df["Date"].duplicated()

    audit_records.append({
        "Name": stock_name,
        "Ticker": ticker,
        "LSEG_RIC": lseg_ric,
        "Country": country,
        "Rows": len(stock_df),
        "Start_Date": stock_df["Date"].min(),
        "End_Date": stock_df["Date"].max(),
        "Missing_OHLCV": int(missing_ohlcv.sum()),
        "Invalid_Prices": int(invalid_prices.sum()),
        "Negative_Volume": int(invalid_volume.sum()),
        "Invalid_High_Low": int(invalid_high_low.sum()),
        "OHLC_Inconsistencies": int(invalid_ohlc.sum()),
        "Duplicate_Dates": int(duplicate_dates.sum())
    })

    all_stocks.append(stock_df)

# Combine all extracted stock records
lseg_data = pd.concat(
    all_stocks,
    ignore_index=True
)

lseg_data = (
    lseg_data
    .sort_values(["Ticker", "Date"])
    .reset_index(drop=True)
)

audit_df = pd.DataFrame(audit_records)

# Save extracted observations and quality report
lseg_data.to_csv(
    PROCESSED_DIR / "lseg_ohlcv_extracted.csv",
    index=False
)

audit_df.to_csv(
    RESULTS_DIR / "lseg_data_quality_audit.csv",
    index=False
)

print("=== LSEG DATA EXTRACTION ===")
print("Stocks:", lseg_data["Ticker"].nunique())
print("Total observations:", len(lseg_data))
print("Date range:",
      lseg_data["Date"].min().date(),
      "to",
      lseg_data["Date"].max().date())

print("\n=== DATA QUALITY AUDIT ===")
display(audit_df)

print("\n=== MARKET SUMMARY ===")
display(
    lseg_data.groupby("Country").agg(
        Stocks=("Ticker", "nunique"),
        Rows=("Date", "size"),
        Start_Date=("Date", "min"),
        End_Date=("Date", "max")
    )
)

print("\n=== SAMPLE HISTORICAL RECORDS ===")
display(lseg_data.head())

assert lseg_data["Ticker"].nunique() == 13

print("\nExtracted data and audit saved.")

=== LSEG DATA EXTRACTION ===
Stocks: 13
Total observations: 37864
Date range: 2015-01-02 to 2026-10-30

=== DATA QUALITY AUDIT ===


,Name,Ticker,LSEG_RIC,Country,Rows,Start_Date,End_Date,Missing_OHLCV,Invalid_Prices,Negative_Volume,Invalid_High_Low,OHLC_Inconsistencies,Duplicate_Dates
0,Affin Bank,5185.KL,AFIN.KL,Malaysia,2896,2015-01-02,2026-10-30,11,0,0,0,0,0
1,Alliance Bank,2488.KL,ALLI.KL,Malaysia,2896,2015-01-02,2026-10-30,9,0,0,0,0,0
2,Ambank,1015.KL,AMMB.KL,Malaysia,2896,2015-01-02,2026-10-30,4,0,0,0,0,0
3,BIMB,5258.KL,BIMB.KL,Malaysia,2896,2015-01-02,2026-10-30,18,0,0,0,0,0
4,CIMB,1023.KL,CIMB.KL,Malaysia,2896,2015-01-02,2026-10-30,0,0,0,0,0,0
5,DBS,D05.SI,DBSM.SI,Singapore,2968,2015-01-02,2026-10-30,0,0,0,0,0,0
6,Hong Leong,5819.KL,HLBB.KL,Malaysia,2896,2015-01-02,2026-10-30,0,0,0,0,0,0
7,Maybank,1155.KL,MBBM.KL,Malaysia,2896,2015-01-02,2026-10-30,0,0,0,0,0,0
8,MBSB,1171.KL,MBSB.KL,Malaysia,2896,2015-01-02,2026-10-30,1,0,0,0,0,0
9,OCBC,O39.SI,OCBC.SI,Singapore,2968,2015-01-02,2026-10-30,0,0,0,0,0,0



=== MARKET SUMMARY ===


,Stocks,Rows,Start_Date,End_Date
Country,,,,
Malaysia,10,28960,2015-01-02,2026-10-30
Singapore,3,8904,2015-01-02,2026-10-30



=== SAMPLE HISTORICAL RECORDS ===


,Date,Open,High,Low,Close,Volume,Ticker,Name,Country,LSEG_RIC
0,2015-01-02,6.54,6.61,6.52,6.54,1314300.0,1015.KL,Ambank,Malaysia,AMMB.KL
1,2015-01-05,6.59,6.59,6.43,6.44,7837100.0,1015.KL,Ambank,Malaysia,AMMB.KL
2,2015-01-06,6.39,6.39,6.09,6.30,6784200.0,1015.KL,Ambank,Malaysia,AMMB.KL
3,2015-01-07,6.30,6.40,6.25,6.33,4363700.0,1015.KL,Ambank,Malaysia,AMMB.KL
4,2015-01-08,6.28,6.43,6.28,6.43,2785000.0,1015.KL,Ambank,Malaysia,AMMB.KL



Extracted data and audit saved.


**Investigate missing OHLCV observations**

In [12]:
import pandas as pd
import numpy as np

OHLCV_COLUMNS = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume"
]

# Count missing values for each stock and field
missing_by_stock = (
    lseg_data
    .groupby(["Country", "Name", "Ticker"])[OHLCV_COLUMNS]
    .apply(lambda x: x.isna().sum())
    .reset_index()
)

missing_by_stock["Affected_Rows"] = (
    lseg_data
    .assign(
        Has_Missing=lseg_data[OHLCV_COLUMNS]
        .isna().any(axis=1)
    )
    .groupby(["Country", "Name", "Ticker"])["Has_Missing"]
    .sum()
    .values
)

print("=== MISSING VALUES BY STOCK AND COLUMN ===")

display(
    missing_by_stock[
        missing_by_stock["Affected_Rows"] > 0
    ]
)

# Extract rows containing at least one missing OHLCV field
missing_rows = lseg_data.loc[
    lseg_data[OHLCV_COLUMNS].isna().any(axis=1)
].copy()

# Identify which fields are missing
missing_rows["Missing_Fields"] = (
    missing_rows[OHLCV_COLUMNS]
    .isna()
    .apply(
        lambda row: ", ".join(
            row.index[row].tolist()
        ),
        axis=1
    )
)

print("\n=== TOTAL MISSING OBSERVATIONS ===")
print("Affected rows:", len(missing_rows))

print("\n=== MISSING FIELD PATTERNS ===")

display(
    missing_rows["Missing_Fields"]
    .value_counts()
    .rename_axis("Missing_Fields")
    .reset_index(name="Rows")
)

print("\n=== AFFECTED HISTORICAL RECORDS ===")

display(
    missing_rows[
        [
            "Date",
            "Name",
            "Ticker",
            "Open",
            "High",
            "Low",
            "Close",
            "Volume",
            "Missing_Fields"
        ]
    ].sort_values(["Date", "Ticker"])
)

# Identify dates affecting multiple banks
missing_dates = (
    missing_rows
    .groupby("Date")
    .agg(
        Affected_Banks=("Ticker", "nunique"),
        Rows=("Ticker", "size")
    )
    .sort_values(
        ["Affected_Banks", "Date"],
        ascending=[False, True]
    )
)

print("\n=== DATES WITH MULTIPLE AFFECTED BANKS ===")

display(
    missing_dates[
        missing_dates["Affected_Banks"] > 1
    ]
)

# Check zero-volume observations
zero_volume = lseg_data.loc[
    lseg_data["Volume"] == 0,
    ["Date", "Name", "Ticker", "Close", "Volume"]
].copy()

print("\n=== ZERO-VOLUME OBSERVATIONS ===")
print("Rows with zero volume:", len(zero_volume))

display(zero_volume.head(30))

# Save investigation outputs
missing_by_stock.to_csv(
    RESULTS_DIR / "lseg_missing_values_by_stock.csv",
    index=False
)

missing_rows.to_csv(
    RESULTS_DIR / "lseg_missing_observations.csv",
    index=False
)

missing_dates.to_csv(
    RESULTS_DIR / "lseg_missing_dates.csv"
)

zero_volume.to_csv(
    RESULTS_DIR / "lseg_zero_volume_observations.csv",
    index=False
)

print("\nCompleted.")

=== MISSING VALUES BY STOCK AND COLUMN ===


,Country,Name,Ticker,Open,High,Low,Close,Volume,Affected_Rows
0,Malaysia,Affin Bank,5185.KL,11,11,11,11,11,11
1,Malaysia,Alliance Bank,2488.KL,9,9,9,9,9,9
2,Malaysia,Ambank,1015.KL,4,4,4,4,4,4
3,Malaysia,BIMB,5258.KL,18,18,18,18,18,18
6,Malaysia,MBSB,1171.KL,1,1,1,1,1,1
8,Malaysia,Public Bank,1295.KL,2,2,2,2,2,2
9,Malaysia,RHB,1066.KL,21,21,21,21,21,21



=== TOTAL MISSING OBSERVATIONS ===
Affected rows: 66

=== MISSING FIELD PATTERNS ===


,Missing_Fields,Rows
0,"Open, High, Low, Close, Volume",66



=== AFFECTED HISTORICAL RECORDS ===


,Date,Name,Ticker,Open,High,Low,Close,Volume,Missing_Fields
6139,2016-05-31,RHB,1066.KL,NaN,NaN,NaN,NaN,NaN,"Open, High, Low, Close, Volume"
6140,2016-06-01,RHB,1066.KL,NaN,NaN,NaN,NaN,NaN,"Open, High, Low, Close, Volume"
6141,2016-06-02,RHB,1066.KL,NaN,NaN,NaN,NaN,NaN,"Open, High, Low, Close, Volume"
6142,2016-06-03,RHB,1066.KL,NaN,NaN,NaN,NaN,NaN,"Open, High, Low, Close, Volume"
6143,2016-06-06,RHB,1066.KL,NaN,NaN,NaN,NaN,NaN,"Open, High, Low, Close, Volume"
...,...,...,...,...,...,...,...,...,...
24822,2021-10-05,BIMB,5258.KL,NaN,NaN,NaN,NaN,NaN,"Open, High, Low, Close, Volume"
24823,2021-10-06,BIMB,5258.KL,NaN,NaN,NaN,NaN,NaN,"Open, High, Low, Close, Volume"
24824,2021-10-07,BIMB,5258.KL,NaN,NaN,NaN,NaN,NaN,"Open, High, Low, Close, Volume"
16871,2024-10-09,Public Bank,1295.KL,NaN,NaN,NaN,NaN,NaN,"Open, High, Low, Close, Volume"



=== DATES WITH MULTIPLE AFFECTED BANKS ===


,Affected_Banks,Rows
Date,,
2016-06-27,2,2
2017-06-01,2,2
2017-08-22,2,2



=== ZERO-VOLUME OBSERVATIONS ===
Rows with zero volume: 0


,Date,Name,Ticker,Close,Volume



Completed.


**Analyse consecutive missing-data periods**

In [14]:
import pandas as pd
import numpy as np

OHLCV_COLUMNS = ["Open", "High", "Low", "Close", "Volume"]

gap_records = []

for ticker, stock in lseg_data.groupby("Ticker"):

    stock = (
        stock.sort_values("Date")
        .reset_index(drop=True)
    )

    missing_mask = stock[OHLCV_COLUMNS].isna().all(axis=1)

    # Identify consecutive missing-record groups
    group_ids = (
        missing_mask.ne(missing_mask.shift(fill_value=False))
        .cumsum()
    )

    for group_id in group_ids[missing_mask].unique():

        missing_positions = np.flatnonzero(
            (group_ids == group_id).to_numpy()
        )

        first_pos = int(missing_positions[0])
        last_pos = int(missing_positions[-1])

        before_pos = first_pos - 1
        after_pos = last_pos + 1

        before = (
            stock.iloc[before_pos]
            if before_pos >= 0
            else None
        )

        after = (
            stock.iloc[after_pos]
            if after_pos < len(stock)
            else None
        )

        before_close = (
            before["Close"] if before is not None else np.nan
        )

        after_close = (
            after["Close"] if after is not None else np.nan
        )

        # Calculate change across the missing period
        gap_return = (
            (after_close / before_close - 1) * 100
            if pd.notna(before_close)
            and pd.notna(after_close)
            and before_close > 0
            else np.nan
        )

        gap_records.append({
            "Ticker": ticker,
            "Name": stock.loc[first_pos, "Name"],
            "Country": stock.loc[first_pos, "Country"],
            "Gap_Start": stock.loc[first_pos, "Date"],
            "Gap_End": stock.loc[last_pos, "Date"],
            "Missing_Rows": len(missing_positions),
            "Previous_Date": (
                before["Date"] if before is not None else pd.NaT
            ),
            "Previous_Close": before_close,
            "Next_Date": (
                after["Date"] if after is not None else pd.NaT
            ),
            "Next_Close": after_close,
            "Return_Across_Gap_Pct": gap_return
        })

gap_df = (
    pd.DataFrame(gap_records)
    .sort_values(["Gap_Start", "Ticker"])
    .reset_index(drop=True)
)

print("=== MISSING-DATA PERIOD ANALYSIS ===")

print("Total missing observations:", len(missing_rows))
print("Number of missing-data periods:", len(gap_df))
print("Missing rows accounted for:", gap_df["Missing_Rows"].sum())

display(gap_df)

# Verify all completely missing rows were accounted for
assert gap_df["Missing_Rows"].sum() == len(missing_rows)

print("\n=== LONGEST MISSING PERIODS ===")

display(
    gap_df.sort_values(
        "Missing_Rows",
        ascending=False
    ).head(10)
)

print("\n=== LARGE PRICE CHANGES ACROSS GAPS ===")

large_gap_changes = gap_df[
    gap_df["Return_Across_Gap_Pct"].abs() >= 5
]

display(large_gap_changes)

# Save missing-period analysis
gap_df.to_csv(
    RESULTS_DIR / "lseg_missing_data_periods.csv",
    index=False
)

print("\nCompleted.")

=== MISSING-DATA PERIOD ANALYSIS ===
Total missing observations: 66
Number of missing-data periods: 18
Missing rows accounted for: 66


,Ticker,Name,Country,Gap_Start,Gap_End,Missing_Rows,Previous_Date,Previous_Close,Next_Date,Next_Close,Return_Across_Gap_Pct
0,1066.KL,RHB,Malaysia,2016-05-31,2016-06-27,19,2016-05-30,4.640002,2016-06-28,4.840000,4.310300
1,5258.KL,BIMB,Malaysia,2016-06-27,2016-06-27,1,2016-06-24,2.971616,2016-06-28,2.964131,-0.251883
2,5258.KL,BIMB,Malaysia,2016-11-23,2016-11-23,1,2016-11-22,3.233598,2016-11-24,3.158746,-2.314821
3,5258.KL,BIMB,Malaysia,2016-12-20,2016-12-20,1,2016-12-19,3.218627,2016-12-21,3.218627,0.000000
4,1015.KL,Ambank,Malaysia,2017-06-01,2017-06-01,1,2017-05-31,5.210000,2017-06-02,5.090000,-2.303263
5,1066.KL,RHB,Malaysia,2017-06-01,2017-06-01,1,2017-05-31,5.390000,2017-06-02,5.190000,-3.710575
6,1015.KL,Ambank,Malaysia,2017-08-22,2017-08-22,1,2017-08-21,4.700000,2017-08-23,4.590000,-2.340426
7,1066.KL,RHB,Malaysia,2017-08-22,2017-08-22,1,2017-08-21,4.880000,2017-08-23,5.070000,3.893443
8,2488.KL,Alliance Bank,Malaysia,2017-09-12,2017-09-25,9,2017-09-11,3.770936,2017-09-26,3.741704,-0.775192
9,5258.KL,BIMB,Malaysia,2017-09-27,2017-09-28,2,2017-09-26,3.256053,2017-09-29,3.285994,0.919549



=== LONGEST MISSING PERIODS ===


,Ticker,Name,Country,Gap_Start,Gap_End,Missing_Rows,Previous_Date,Previous_Close,Next_Date,Next_Close,Return_Across_Gap_Pct
0,1066.KL,RHB,Malaysia,2016-05-31,2016-06-27,19,2016-05-30,4.640002,2016-06-28,4.840000,4.310300
12,5185.KL,Affin Bank,Malaysia,2018-01-16,2018-01-30,11,2018-01-15,2.225067,2018-02-02,2.277836,2.371569
16,5258.KL,BIMB,Malaysia,2021-09-27,2021-10-07,9,2021-09-24,3.016528,2021-10-08,3.040000,0.778113
8,2488.KL,Alliance Bank,Malaysia,2017-09-12,2017-09-25,9,2017-09-11,3.770936,2017-09-26,3.741704,-0.775192
11,5258.KL,BIMB,Malaysia,2017-11-14,2017-11-15,2,2017-11-13,3.241083,2017-11-16,3.218627,-0.692855
9,5258.KL,BIMB,Malaysia,2017-09-27,2017-09-28,2,2017-09-26,3.256053,2017-09-29,3.285994,0.919549
17,1295.KL,Public Bank,Malaysia,2024-10-09,2024-10-10,2,2024-10-08,4.570000,2024-10-11,4.310000,-5.689278
14,1015.KL,Ambank,Malaysia,2021-03-01,2021-03-02,2,2021-02-26,3.160000,2021-03-03,2.800000,-11.392405
1,5258.KL,BIMB,Malaysia,2016-06-27,2016-06-27,1,2016-06-24,2.971616,2016-06-28,2.964131,-0.251883
2,5258.KL,BIMB,Malaysia,2016-11-23,2016-11-23,1,2016-11-22,3.233598,2016-11-24,3.158746,-2.314821



=== LARGE PRICE CHANGES ACROSS GAPS ===


,Ticker,Name,Country,Gap_Start,Gap_End,Missing_Rows,Previous_Date,Previous_Close,Next_Date,Next_Close,Return_Across_Gap_Pct
10,1171.KL,MBSB,Malaysia,2017-11-06,2017-11-06,1,2017-11-03,1.11,2017-11-07,1.17,5.405405
14,1015.KL,Ambank,Malaysia,2021-03-01,2021-03-02,2,2021-02-26,3.16,2021-03-03,2.80,-11.392405
17,1295.KL,Public Bank,Malaysia,2024-10-09,2024-10-10,2,2024-10-08,4.57,2024-10-11,4.31,-5.689278



Completed.


**Clean LSEG observations and preserve gap boundaries**

In [15]:
import pandas as pd
import numpy as np

OHLCV_COLUMNS = ["Open", "High", "Low", "Close", "Volume"]

cleaned_stocks = []
gap_boundary_records = []

for ticker, stock in lseg_data.groupby("Ticker"):

    stock = stock.sort_values("Date").copy()
    stock = stock.reset_index(drop=True)

    # Identify complete and partial missing records
    missing_all = stock[OHLCV_COLUMNS].isna().all(axis=1)
    missing_any = stock[OHLCV_COLUMNS].isna().any(axis=1)

    assert not (missing_any & ~missing_all).any(), (
        f"Partial OHLCV missing values found: {ticker}"
    )

    # Identify the start of each missing-data period
    gap_start = (
        missing_all
        & ~missing_all.shift(fill_value=False)
    )

    # Assign a new segment after each missing-data period
    stock["Segment_ID"] = gap_start.cumsum().astype(int)

    # Identify valid observations adjacent to missing periods
    stock["Before_Gap"] = (
        ~missing_all
        & missing_all.shift(-1, fill_value=False)
    )

    stock["After_Gap"] = (
        ~missing_all
        & missing_all.shift(fill_value=False)
    )

    # Remove completely missing OHLCV records
    valid = stock.loc[~missing_all].copy()

    # Calculate historical returns within uninterrupted segments
    valid["returns"] = (
        valid.groupby("Segment_ID")["Close"]
        .pct_change(fill_method=None)
    )

    # Check the number of valid segments
    segment_count = valid["Segment_ID"].nunique()

    gap_boundary_records.append({
        "Ticker": ticker,
        "Name": valid["Name"].iloc[0],
        "Original_Rows": len(stock),
        "Removed_Rows": int(missing_all.sum()),
        "Clean_Rows": len(valid),
        "Segments": segment_count,
        "Before_Gap_Rows": int(valid["Before_Gap"].sum()),
        "After_Gap_Rows": int(valid["After_Gap"].sum()),
        "Missing_Returns": int(valid["returns"].isna().sum())
    })

    cleaned_stocks.append(valid)

# Combine all valid observations
lseg_clean = pd.concat(
    cleaned_stocks,
    ignore_index=True
)

lseg_clean = (
    lseg_clean
    .sort_values(["Ticker", "Date"])
    .reset_index(drop=True)
)

cleaning_audit = pd.DataFrame(gap_boundary_records)

# Validate extraction and cleaning
assert len(lseg_clean) == 37864 - 66
assert lseg_clean["Ticker"].nunique() == 13
assert not lseg_clean[OHLCV_COLUMNS].isna().any().any()
assert not lseg_clean.duplicated(["Ticker", "Date"]).any()

assert cleaning_audit["Removed_Rows"].sum() == 66
assert cleaning_audit["Before_Gap_Rows"].sum() == 18
assert cleaning_audit["After_Gap_Rows"].sum() == 18

# Verify that returns immediately after missing periods are undefined
assert lseg_clean.loc[
    lseg_clean["After_Gap"], "returns"
].isna().all()

# Save cleaned data and audit
lseg_clean.to_csv(
    PROCESSED_DIR / "lseg_ohlcv_clean.csv",
    index=False
)

cleaning_audit.to_csv(
    RESULTS_DIR / "lseg_cleaning_audit.csv",
    index=False
)

print("=== LSEG DATA CLEANING SUMMARY ===")

print("Original observations:", len(lseg_data))
print("Removed observations:", len(lseg_data) - len(lseg_clean))
print("Clean observations:", len(lseg_clean))
print("Stocks:", lseg_clean["Ticker"].nunique())
print("Date range:",
      lseg_clean["Date"].min().date(),
      "to",
      lseg_clean["Date"].max().date())

print("\n=== CLEANING AUDIT ===")
display(cleaning_audit)

print("\n=== OBSERVATIONS AFTER MISSING PERIODS ===")

display(
    lseg_clean.loc[
        lseg_clean["After_Gap"],
        [
            "Date",
            "Ticker",
            "Close",
            "Segment_ID",
            "returns",
            "After_Gap"
        ]
    ].head(20)
)

print("\nCompleted. Cleaned dataset saved.")

=== LSEG DATA CLEANING SUMMARY ===
Original observations: 37864
Removed observations: 66
Clean observations: 37798
Stocks: 13
Date range: 2015-01-02 to 2026-10-30

=== CLEANING AUDIT ===


,Ticker,Name,Original_Rows,Removed_Rows,Clean_Rows,Segments,Before_Gap_Rows,After_Gap_Rows,Missing_Returns
0,1015.KL,Ambank,2896,4,2892,4,3,3,4
1,1023.KL,CIMB,2896,0,2896,1,0,0,1
2,1066.KL,RHB,2896,21,2875,4,3,3,4
3,1155.KL,Maybank,2896,0,2896,1,0,0,1
4,1171.KL,MBSB,2896,1,2895,2,1,1,2
5,1295.KL,Public Bank,2896,2,2894,2,1,1,2
6,2488.KL,Alliance Bank,2896,9,2887,2,1,1,2
7,5185.KL,Affin Bank,2896,11,2885,2,1,1,2
8,5258.KL,BIMB,2896,18,2878,9,8,8,9
9,5819.KL,Hong Leong,2896,0,2896,1,0,0,1



=== OBSERVATIONS AFTER MISSING PERIODS ===


,Date,Ticker,Close,Segment_ID,returns,After_Gap
593,2017-06-02,1015.KL,5.090000,1,NaN,True
647,2017-08-23,1015.KL,4.590000,2,NaN,True
1505,2021-03-03,1015.KL,2.800000,3,NaN,True
6135,2016-06-28,1066.KL,4.840000,1,NaN,True
6362,2017-06-02,1066.KL,5.190000,2,NaN,True
6416,2017-08-23,1066.KL,5.070000,3,NaN,True
12256,2017-11-07,1171.KL,1.170000,1,NaN,True
16845,2024-10-11,1295.KL,4.310000,1,NaN,True
18008,2017-09-26,2488.KL,3.741704,1,NaN,True
20980,2018-02-02,5185.KL,2.277836,1,NaN,True



Completed. Cleaned dataset saved.


**Validate cleaned LSEG dataset**

In [16]:
import pandas as pd
import numpy as np

# Load saved cleaned dataset
clean_path = PROCESSED_DIR / "lseg_ohlcv_clean.csv"

df = pd.read_csv(
    clean_path,
    parse_dates=["Date"]
)

OHLCV_COLUMNS = ["Open", "High", "Low", "Close", "Volume"]

# Validate dataset structure
assert len(df) == 37798
assert df["Ticker"].nunique() == 13
assert not df[OHLCV_COLUMNS].isna().any().any()
assert not df.duplicated(["Ticker", "Date"]).any()

# Validate price and volume values
assert (df[["Open", "High", "Low", "Close"]] > 0).all().all()
assert (df["Volume"] >= 0).all()

assert (df["High"] >= df["Low"]).all()
assert (df["High"] >= df[["Open", "Close"]].max(axis=1)).all()
assert (df["Low"] <= df[["Open", "Close"]].min(axis=1)).all()

# Validate chronological ordering
assert df.equals(
    df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
)

# Validate return calculations and segment boundaries
df["Expected_Return"] = (
    df.groupby(["Ticker", "Segment_ID"])["Close"]
    .pct_change(fill_method=None)
)

assert np.allclose(
    df["returns"],
    df["Expected_Return"],
    equal_nan=True
)

assert df.loc[df["After_Gap"], "returns"].isna().all()

# Summarise cleaned stock coverage
stock_summary = (
    df.groupby(["Country", "Name", "Ticker"])
    .agg(
        Observations=("Date", "size"),
        Start_Date=("Date", "min"),
        End_Date=("Date", "max"),
        Segments=("Segment_ID", "nunique")
    )
    .reset_index()
)

print("=== FINAL DATASET VALIDATION ===")

print("Total observations:", len(df))
print("Total stocks:", df["Ticker"].nunique())
print("Missing OHLCV values:", int(df[OHLCV_COLUMNS].isna().sum().sum()))
print("Duplicate stock-date records:", int(df.duplicated(["Ticker", "Date"]).sum()))
print("Gap boundaries:", int(df["After_Gap"].sum()))

display(stock_summary)

# Flag large valid daily returns for review
large_returns = df.loc[
    df["returns"].abs() >= 0.20,
    ["Date", "Ticker", "Name", "Close", "returns"]
].copy()

large_returns["Return_Pct"] = large_returns["returns"] * 100

print("\n=== DAILY RETURNS OF AT LEAST 20% ===")
print("Observations flagged:", len(large_returns))

display(
    large_returns.sort_values(
        "Return_Pct",
        key=lambda x: x.abs(),
        ascending=False
    )
)

# Save validation reports
stock_summary.to_csv(
    RESULTS_DIR / "lseg_final_stock_summary.csv",
    index=False
)

large_returns.to_csv(
    RESULTS_DIR / "lseg_large_daily_returns.csv",
    index=False
)

print("\nCompleted.")

=== FINAL DATASET VALIDATION ===
Total observations: 37798
Total stocks: 13
Missing OHLCV values: 0
Duplicate stock-date records: 0
Gap boundaries: 18


,Country,Name,Ticker,Observations,Start_Date,End_Date,Segments
0,Malaysia,Affin Bank,5185.KL,2885,2015-01-02,2026-10-30,2
1,Malaysia,Alliance Bank,2488.KL,2887,2015-01-02,2026-10-30,2
2,Malaysia,Ambank,1015.KL,2892,2015-01-02,2026-10-30,4
3,Malaysia,BIMB,5258.KL,2878,2015-01-02,2026-10-30,9
4,Malaysia,CIMB,1023.KL,2896,2015-01-02,2026-10-30,1
5,Malaysia,Hong Leong,5819.KL,2896,2015-01-02,2026-10-30,1
6,Malaysia,MBSB,1171.KL,2895,2015-01-02,2026-10-30,2
7,Malaysia,Maybank,1155.KL,2896,2015-01-02,2026-10-30,1
8,Malaysia,Public Bank,1295.KL,2894,2015-01-02,2026-10-30,2
9,Malaysia,RHB,1066.KL,2875,2015-01-02,2026-10-30,4



=== DAILY RETURNS OF AT LEAST 20% ===
Observations flagged: 1


,Date,Ticker,Name,Close,returns,Return_Pct
11921,2016-06-21,1171.KL,MBSB,0.795,-0.231884,-23.188406



Completed.


**Inspect MBSB prices around the rights issue**

In [17]:
mbsb = df.loc[
    (df["Ticker"] == "1171.KL")
    & df["Date"].between("2016-06-13", "2016-06-30")
].copy()

# Calculate the previous closing price
mbsb["Previous_Close"] = mbsb["Close"].shift(1)

# Convert returns to percentages
mbsb["Return_Pct"] = mbsb["returns"] * 100

# Identify the corporate action date
mbsb["Rights_Ex_Date"] = (
    mbsb["Date"] == pd.Timestamp("2016-06-21")
)

print("=== MBSB RIGHTS ISSUE PRICE INSPECTION ===")

display(
    mbsb[
        [
            "Date",
            "Open",
            "High",
            "Low",
            "Close",
            "Previous_Close",
            "Volume",
            "Return_Pct",
            "Rights_Ex_Date"
        ]
    ]
)

# Save the event inspection
mbsb.to_csv(
    RESULTS_DIR / "mbsb_rights_issue_20160621.csv",
    index=False
)

print("\nCompleted.")

=== MBSB RIGHTS ISSUE PRICE INSPECTION ===


,Date,Open,High,Low,Close,Previous_Close,Volume,Return_Pct,Rights_Ex_Date
11915,2016-06-13,1.122056,1.122056,1.102711,1.102711,NaN,1.327937e+06,-1.724067,False
11916,2016-06-14,1.112384,1.112384,1.093038,1.102711,1.102711,1.223004e+06,0.000000,False
11917,2016-06-15,1.102711,1.102711,1.083365,1.093038,1.102711,2.483950e+06,-0.877202,False
11918,2016-06-16,1.093038,1.093038,1.064019,1.064019,1.093038,2.309028e+06,-2.654894,False
11919,2016-06-17,1.064019,1.083365,1.044673,1.054346,1.064019,3.396603e+06,-0.909100,False
11920,2016-06-20,1.054346,1.054346,1.015655,1.035000,1.054346,6.272886e+06,-1.834882,False
11921,2016-06-21,0.820000,0.820000,0.775000,0.795000,1.035000,1.090950e+07,-23.188406,True
11922,2016-06-23,0.795000,0.795000,0.785000,0.785000,0.795000,2.011600e+06,-1.257862,False
11923,2016-06-24,0.785000,0.790000,0.765000,0.770000,0.785000,3.938200e+06,-1.910828,False
11924,2016-06-27,0.755000,0.755000,0.710000,0.720000,0.770000,1.605250e+07,-6.493506,False



Completed.
